# Project 3 — Connect-4 Policy Gradient Training (Q1–Q3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Stiles-Clements1/connect4-rl-arena/blob/main/notebooks/project3_pg_training.ipynb)

This notebook implements **Questions 1–3** of the assignment:

1. Select a pretrained Connect-4 network as **M1** (Stiles Transformer).
2. Improve M1 via **policy gradient self-play** against randomly chosen M2 opponents.
3. Maintain a growing **opponent pool** that periodically absorbs frozen copies of the updated M1.

Heavy logic lives in `src/`; this notebook stays thin — it just wires things together and shows results.

**Runs on both Colab and local.** Cell 1 detects the environment automatically — no edits needed.

> This notebook is the Q1–Q3 baseline PG trainer. For the stronger SAC trainer used for the final submission,
> see [`sac_training.ipynb`](./sac_training.ipynb).

In [ ]:
# Cell 1 — Environment setup + imports (Colab OR local).
#
# Detects Colab vs local, clones/pulls the repo on Colab, walks up to the
# repo root on local, adds it to sys.path, and imports the src/ modules.

import os
import sys
import random
import subprocess
from pathlib import Path

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

IS_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Stiles-Clements1/connect4-rl-arena.git'

if IS_COLAB:
    REPO_ROOT = Path('/content/connect4-rl-arena')
    if not REPO_ROOT.exists():
        print(f'Cloning {REPO_URL} → {REPO_ROOT}')
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_ROOT)], check=True)
    else:
        print(f'Repo already at {REPO_ROOT}; pulling latest main.')
        subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'], check=False)
    subprocess.run(['pip', 'install', '-q', 'wandb', 'tqdm', 'ipywidgets'], check=False)
else:
    here = Path.cwd().resolve()
    REPO_ROOT = None
    for p in [here, *here.parents]:
        if (p / 'src' / 'game_engine.py').exists():
            REPO_ROOT = p
            break
    if REPO_ROOT is None:
        raise RuntimeError(
            f'Could not find repo root starting from {here}. '
            f'Run the notebook from inside the cloned connect4-rl-arena repo.'
        )
    print(f'Local repo root: {REPO_ROOT}')

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

# Purge any cached src.* modules so edits get picked up on re-run.
for _m in [k for k in sys.modules if k == 'src' or k.startswith('src.')]:
    del sys.modules[_m]

from src import config, game_engine, model_loader, opponent_pool, pg_trainer

print(f'\nTensorFlow = {tf.__version__}  |  IS_COLAB = {IS_COLAB}')
print(f'Project root = {REPO_ROOT}')

In [ ]:
# Cell 2 (optional) — Weights & Biases login.
#
# pg_trainer.py calls `wandb.init(...)` so if you want live loss curves
# streamed to wandb.ai, run this cell once to authenticate. Safe to skip —
# training will still run and print metrics to stdout, the wandb call just
# falls back to offline mode or anonymous logging.
#
# Colab: pastes your token interactively. Local: reads ~/.netrc.
# If you haven't set up wandb and don't care about it, skip this cell.

# import wandb
# wandb.login()

In [ ]:
# Cell 3 — Set random seeds for reproducibility.
random.seed(config.SEED)
np.random.seed(config.SEED)
tf.random.set_seed(config.SEED)

print(f'Seeds set to {config.SEED}')
print()
print('Key hyperparameters:')
print(f'  GAMES_PER_GROUP     = {config.GAMES_PER_GROUP}')
print(f'  BATCH_SIZE          = {config.BATCH_SIZE}')
print(f'  GAMMA               = {config.GAMMA}')
print(f'  LEARNING_RATE       = {config.LEARNING_RATE}')
print(f'  NUM_GROUPS          = {config.NUM_GROUPS}')
print(f'  RANDOM_INIT_MOVES   = {config.RANDOM_INIT_MOVES}')
print(f'  POOL_CAP            = {config.POOL_CAP}')
print(f'  POOL_ADD_INTERVAL   = {config.POOL_ADD_INTERVAL}')
print(f'  CHECKPOINT_INTERVAL = {config.CHECKPOINT_INTERVAL}')

In [ ]:
# Cell 4 — Load all models.
# This may take ~30-60 seconds on first run (Keras/TF model loading).
models = model_loader.load_all_models()

m1_wrapper = models['m1']
print(f"\nM1 (to be trained): {m1_wrapper.name}  |  encoding={m1_wrapper.encoding}")

In [ ]:
# Cell 5 — Build the opponent pool.
# All models except M1 itself seed the pool as permanent (never-removed) opponents.
initial_m2 = [wrapper for key, wrapper in models.items() if key != 'm1']
pool = opponent_pool.OpponentPool(initial_m2)

print(pool)

In [ ]:
# Cell 6 — Run the policy gradient training loop.
#
# Progress is printed every group; M1 is checkpointed every CHECKPOINT_INTERVAL
# groups into `checkpoints/m1_group_XXXX.keras`. The full log is saved to
# `logs/training_log.json`.
log = pg_trainer.train(m1_wrapper, pool)

In [ ]:
# Cell 7 — Plot training results.
groups    = np.array(log['group'])
losses    = np.array(log['loss'])
win_rates = np.array(log['win_rate'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(groups, losses, color='steelblue', linewidth=0.8, alpha=0.7)
ax1.set_xlabel('Training group')
ax1.set_ylabel('PG loss  (−E[G · log π])')
ax1.set_title('Policy Gradient Loss')
ax1.grid(True, alpha=0.3)

WINDOW = 20
ax2.plot(groups, win_rates, alpha=0.25, color='orange', linewidth=0.8, label='raw')
if len(win_rates) >= WINDOW:
    smoothed = np.convolve(win_rates, np.ones(WINDOW) / WINDOW, mode='valid')
    ax2.plot(groups[WINDOW - 1:], smoothed, color='darkorange',
             linewidth=2, label=f'{WINDOW}-group avg')
ax2.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50% baseline')
ax2.set_xlabel('Training group')
ax2.set_ylabel('Win rate vs M2')
ax2.set_title('M1 Win Rate During Training')
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(str(config.LOG_DIR), 'training_curves.png')
plt.savefig(fig_path, dpi=150)
plt.show()

if len(win_rates) >= WINDOW:
    print(f'Final {WINDOW}-group average win rate: {smoothed[-1]:.1%}')
print(f'Figure saved to {fig_path}')

---

## ⚠️ Promote this PG-trained M1 to the tournament model (manual)

If this PG run produced a model worth submitting, copy the latest checkpoint
into `RL models/` using the cell below.

- The cell is commented out so re-running every cell cannot accidentally
  clobber a teammate's already-deployed model.
- It picks the latest `m1_group_XXXX.keras` under `checkpoints/`.
- Destination file: `RL models/pg_m1.keras` (change the name if you want to
  keep multiple PG variants).

In [ ]:
# # Cell 8 (commented) — promote the latest PG checkpoint to RL models/.
#
# import shutil
# from pathlib import Path
#
# ckpt_dir = REPO_ROOT / 'checkpoints'
# candidates = sorted(ckpt_dir.glob('m1_group_*.keras'))
# assert candidates, f'No m1_group_*.keras checkpoints in {ckpt_dir}'
# src = candidates[-1]
# dst = REPO_ROOT / 'RL models' / 'pg_m1.keras'
# dst.parent.mkdir(parents=True, exist_ok=True)
#
# old_size = dst.stat().st_size if dst.exists() else 0
# shutil.copy(src, dst)
# print(f'Copied {src.name} → {dst.relative_to(REPO_ROOT)}')
# print(f'  old size: {old_size / 1024**2:6.2f} MB')
# print(f'  new size: {dst.stat().st_size / 1024**2:6.2f} MB')